<center><font size="4"><b>Technologia Mowy 2026</b></font></center>
<br />
<center><font size="6"><b><u>Lab 2: Model traktu głosowego źródło - filtr</u></b></font></center>
<br />
<br />

<center><b>Notebook version:</b> 0.6 (21.10.2026)</center>
<center>Copyright Marcin Witkowski, AGH, 2026</center>


# Cele ćwiczenia

W ramach tego notatnika:

*   Zapoznasz się z interfejsami `scipy.signal.ShortTimeFFT`
*   Określisz parametry krótkoczasowej transformacji Fouriera (STFT) na potrzeby analizy sygnału mowy
*   Zaimplementujesz model źródło-filtr tworząc tym samym podstawowy syntezator samogłosek



# Wstęp

W  większości nawet najbardziej nowoczesnych rozwiązań z zakresu przetwarzania sygnału generowanego przez człowieka (mowy, śpiewu) podstawowym narzędziem jest spektrogram. Nauczyciele śpiewu analizując proporcje między formantami widocznymi na spekrogramie są w stanie określić poprawność śpiewu. W instytutach kryminalistyki specjaliści na podstawie spektrogramów określają podobieństwo głosu. W systemach rozpoznawania mowy i mówcy najczęściej poza  przetwarzaniem wstępnym, pierwszym (a czasami jedynym) krokiem jest przejście do dziedziny czasowo-częstotliwościowej za pomocą STFT.





# 1. Kalibracja STFT

Aby posługiwać się poprawnie implementacjami dostępnymi w Pythonie należy najpierw dokładnie sprawdzić ich działanie. Poniższy kod demonstruje na przykładzie prostego sygnału sinusoidalnego działanie metod z pakietu `scipy.signal` do obliczania STFT i spektrogramu.

Celem tej części jest również motywacja do zaglądania do dokumentacji lub samego kodu tak by używane narzędzia programistyczne działały zgodnie z oczekiwaniami.


## Zadanie 1
W związku z tym:


1.   (0.5p) Zmodyfikuj poniższy kod (parametry funkcji oraz same zmienne) tak by obliczyć zespoloną macierz krótkoczasowej transformacji Fouriera a następnie wyświetlić spektrogram za pomocą przygotowanej funkcji `plot_stft`. Napisz kod przyjmując:
 -  na rysunkach należy otrzymać jeden widoczny prążek dla "środkowej" części STFT o amplitudzie 1 lub 0dB. Zwróć uwagę na poprawne wartości (kolor) mocy na wykresach;
 - przyjmij długość ramki w dziedzienie czasu równą 20 ms.
2.  (0.5p)  W osobnym bloku tekstu poniżej bloku kodu odpowiedz **własnymi słowami** na pytania.

 W efekcie kod z Zadania 1 powinien zwrócić wynik (zwróć uwagę na kalibrację by amplituda i moc zawsze odpowiadały amplitudzie sygnału wejściowego bez przecieku):
 ![spectrogram_ref.png](https://drive.google.com/uc?id=1duKP2YTRC3G4M4bfcO3u-ix7BnftFZ2u)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# Function that plots matrix in linear and logarithmic (dB) scale
def plot_stft(t,f,X,title):
    plt.style.use(['dark_background']) #--> dla osób korzystających z ciemnego motywu
    fig, ax = plt.subplots(1, 2, figsize=(15, 3))
    im1 = ax[0].pcolormesh(t, f, abs(X))
    ax[0].set_title("|"+title+"|")
    ax[0].set_xlabel("Time [sec]")
    ax[0].set_ylabel("Frequency [Hz]")
    fig.colorbar(im1, ax=ax[0])

    eps = 2**(-15)
    X[X==0] = eps
    im1 = ax[1].pcolormesh(t, f, 20*np.log10(np.abs(X)), vmin=20*np.log10(eps))
    ax[1].set_title("20*log10(|"+ title +"|)")
    ax[1].set_xlabel("Time [sec]")
    ax[1].set_ylabel("Frequency [Hz]")
    fig.colorbar(im1, ax=ax[1])
    fig.tight_layout()
    plt.show()


import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[1]))
from scipy.signal import ShortTimeFFT

from utils.plotting import preview_stft

# Kalibrujący sygnał wejściowy
fs = 16000
tx = np.arange(0,0.5-1/fs,1/fs)


frame_len = int(fs * 0.02) # 20ms
hop = frame_len // 2
x = (2 / frame_len) * np.sin(2*np.pi*1000*tx)


stft = ShortTimeFFT(np.ones(frame_len), hop=hop, fs=fs, fft_mode="onesided")
S = stft.stft(x)
t = stft.t(len(x))
f = stft.f

plot_stft(t, f, S, title='STFT')


# old
# plot_stft(t,f,S,"STFT")


# # preview - fajne
# preview_stft(x, fs)


Edytując ten blok tekstu odpowiedz **własnymi słowami** na pytania:
1.  a) Jakie są wymagane (konieczne w zdefiniowaniu) argumenty  `scipy.signal.ShortTimeFFT()`?<p> b) Co zwraca wywołanie kodu z poprawnie zdefiniowanymi argumentami niezbędnymi?

    <font color=red>a) win, hop, fs, reszta jest po '*' więc nie jest wymagana <p>b) zwraca obiekt klasy ShotTimeFFT, dopiero metoda .stft(x) zwraca macierz </font>

2. a) Jaka jest przyczyna faktu, że pierwsza i ostatnia ramka na spektrogramach wyglądają inaczej niż "środkowa" część?

    <font color=red>a) Przeciek na granicznych ramkach (ze wzgledu na dopisywane zera - chyba)</font>

3. a) Czy funkcja `scipy.signal.ShortTimeFFT.stft(x)` poza obliczeniem FFT dla każdej ramki normalizuje wyjściową macierz? <p> b) Jaką operację należy przeprowadzić by amplitudy prążków STFT zwracanych przez `scipy.signal.ShortTimeFFT.stft(x)` odpowiadały amplitudom sygnału i jakie jest uzasadnienie teoretyczne dla tej operacji?

    <font color=red>a) sama metoda nie, ale skalowanie moze zalezec od parametrow w obiekcie <code>ShortTimeFFT()</code> jak: 
    <ul> 
    <li><code> scale_to="magnitude" </code></li>
    <li><code> fft_mode="onesided2X"   </code></li>
    </ul>
    <p> b) dzielimy przez sume wspolczynnikow okna i podwajamy ampltudy dodatnich czesotliwosci, bo mamy jednostronne widmo

    </font>


# 2. Analiza spektralna sygnału mowy

W tym zadaniu należy znaleźć i uzasadnić na podstawie odpowiednich wykresów optymalną długość okna umożliwiającą oszacowanie zmiany formantów dla poszczególnych fonemów.



## Zadanie 2

Zadanie składa się z następujących etapów:
1. (0.25p) Nagranie krótkiej wypowiedzi  jako plik WAV (np. za pomocą Audacity, może to być jedno/kilka zdań z pliku [zdania_z_Corpory](https://drive.google.com/file/d/1XrkFRp5kXOk74f2KhfB6SJrvwtlSZKBo/view?usp=sharing)). Możliwe jest również użycie już nagranej mowy. Istotne jest by każda grupa analizowała inny sygnał (by uniknąć plagiatu).
Przykład na potrzeby zajęć: https://galaxy.agh.edu.pl/~witkow/audio/example_pl.wav
2. (0.25p) Utworzenie skryptu w bloku kodu poniżej, który:
* wczyta nagrany plik WAV;
* odtworzy prawidłowo wczytany dźwięk wykorzystując widget Audio;
* wykreśli oscylogram (przebieg czasowy sygnału) z prawidłowo podpisanymi osiami (amplituda/czas (w sekundach));
* wykreśli spektrogram dla różnych długości ramkowania (okna) w zakresie 5ms, 10ms, 20ms, 50ms, 100ms, 200ms bez przeskoku.
* dobierając odpowiednio rozmiar okna **oraz** zakładkowanie wyświetli spektrogram o jak najbardziej płynnym przebiegu składowych harmonicznych.
3. (0.5p) W bloku tekstu poniżej należy wskazać oraz uzasadnić (**na podstawie spektrogramów wyrysowanych w poprzednim podpunkcie**) jaka długość okna (rozdzielczość w dziedzinie czasu) jest najbardziej odpowiednia dla analizy sygnału mowy, tj. umożliwia w miarę precyzyjne określenie czasów występowania formantów. Zwróć uwagę, by dobrać zakresy ramek wyświetlanych spektrogramów, tak, by faktycznie wybrany zakres długości okna znajdował potwierdzenie w rysynkach.

In [ ]:
# Miejsce na kod do Zadania 2
import scipy.io.wavfile as wav
from IPython.display import Audio
import matplotlib.pyplot as plt
import numpy as np
from utils.drive import download_data

url = "https://drive.google.com/drive/folders/19LGkO7_FWQ3scqdcX4JPQj2Keb-fILA4?usp=drive_link"
data_dir = download_data(url)
file = data_dir/"Nagranie_do_3_2.wav"
#file = "/workspaces/TM-Grupa-6/Laby/Lab2/KamilKotarba.wav" #zmień plik
fs, x = wav.read(file)
display(Audio(file))

time = np.arange(len(x))/fs
plt.figure()
plt.plot(time,x)
plt.xlabel("Time [s]")
plt.ylabel("Amplitude")
plt.show()

wins_ms = [5, 10, 20, 50, 100, 200]
for win_ms in wins_ms:
    win = int(win_ms * fs/1000)
    plt.figure()
    plt.specgram(x, NFFT=win, Fs=fs, noverlap=0)
    plt.xlabel("Time [s]")
    plt.ylabel("Ferquency [Hz]")
    plt.title(f'Okno: {win_ms}ms')
    plt.colorbar(label='Intensity [dB]')
    plt.show()

plt.figure()
plt.specgram(x, NFFT=512, Fs=fs, noverlap=int(512/2))
plt.xlabel("Time [s]")
plt.ylabel("Ferquency [Hz]")
plt.title('Parametry wybrane: Okno: 512spl; Przeskok: 2')
plt.colorbar(label='Intensity [dB]')
plt.show()

## Wnioski z Zadania 2

<font color=red> Z zaprezentowanych wyżej spektrogramów widać, że najlepsza długość okna pozwalająca na dokładne zaobserwowanie formantów to 50ms.  </color>

# 3. Model źródło-filtr

W poniższym ćwiczeniu korzystając z modelu źródło-filtr zaimplementowany zostanie prosty symulator aparatu mowy zdolny do generowania samogłosek.

## Źródło
Jak sama nazwa wskazuje model składa się ze źródła oraz filtru. Struny głosowe jako narząd produkują dźwięk zawierający w uproszczeniu wszystkie kolejne harmoniczne. W tym przykładzie poniżej zamieszczono 3 przykłady pobudzeń:


*   sygnał prostokątny
*   sygnał piłokształtny
*   sygnał pobudzenia na podstawie modelu Lilijencrants-Fant


**Uwaga**: Odtwarzany sygnał może być głośny!

### Literatura o modelu Liljencrants–Fant (LF):
*   krótko: [Saeed V. Vaseghi, "Multimedia Signal Processing", (2007)](https://tauogibarru.files.wordpress.com/2013/05/saeed_v-_vaseghi_multimedia_signal_processing_tbookfi-org.pdf), rozdz. 14.2.1, str. 486
*   nieco szerzej: [Ingmund Bjorkan, PhD Thesis, "Speech Generation and Modification in
Concatenative Speech Synthesis" (2010)](https://ntnuopen.ntnu.no/ntnu-xmlui/bitstream/handle/11250/2368888/318232_FULLTEXT01.pdf?sequence=1&isAllowed=y), rozdz. 2.1.3, str. 13 - 15
*   artykuł źródłowy: [Fant, G. and Liljencrants, J. and Lin, Q., "A four-parameter model of glottal flow", (1985)](https://citeseerx.ist.psu.edu/viewdoc/download?doi=10.1.1.643.5278&rep=rep1&type=pdf)



In [ ]:
import numpy as np
import scipy.signal as signal
import matplotlib.pyplot as plt
from IPython.display import Audio

def LF_glottal_pulse_diff_Vaseghi(t:np.array, f0:int ,fs:int):
    '''
    Returns derivative of glotal pulse based on Liljencrants-Fant (LF) model for
    time vector 't' with 'f0' frequency (pitch) and sampling rate 'fs'.
    Implementation based on Equation (14.1) from page 487 in
    Saeed V. Vaseghi "Multimedia Signal Processing", 2007
    Marcin Witkowski, AGH 2022
    '''
    T0 = 1 / f0
    Tc = 0.9 * T0
    Te = 0.7 * Tc
    w = 2 * np.pi * f0
    Ee = 1

    alfa = 4*f0                 # dostosowanie by MW
    beta = 3**(f0/100)*1000     # dostosowanie by MW

    E0 = -Ee / (np.exp(alfa*Te) * np.sin(w*Te))
    E1 = Ee / (1 - np.exp(-beta*(Tc - Te)))
    t_period = np.arange(0, T0, 1/fs)

    t1 = t_period < Te
    t2 = [a and b for a, b in zip(t_period >= Te, t_period < Tc)]
    t3 = t_period >= Tc

    v1 = E0 * np.multiply(np.exp(alfa*t_period[t1]),np.sin(w*t_period[t1])) # first stage
    v2 = -E1 * (np.exp(-beta*(t_period[t2]-Te)) - np.exp(-beta*(Tc-Te)))    # second stage
    v3 = np.zeros(t_period[t3].shape)             # third stage - zeros

    v_period = np.concatenate((v1, v2, v3), axis=0)
    N = int(np.ceil(len(t)/len(t_period)))
    v_LF = np.tile(v_period,N)[0:len(t)]
    return v_LF

fs = 16000
f0 = 100    # Ton podstawowy, F0
T = 3       # czas trwania sygnału
t = np.arange(0,T-1/fs,1/fs)

# Wybierz sygnał
# x = 0.5 * signal.square(2 * np.pi * f0 * t)
# x = 0.5 * signal.sawtooth(2 * np.pi * f0 * t)
x = LF_glottal_pulse_diff_Vaseghi(t,f0,fs)

fvec, tvec, SQ = signal.stft(x, fs, nperseg=2048, noverlap=160, nfft=2048)
plot_stft(tvec,fvec[0:200],SQ[0:200],"STFT")
plt.plot(t[0:500], x[0:500]); plt.xlabel("Time [sec]"); plt.title("Slice of the glottal pulse model (source) signal"); plt.grid();
plt.show()
f, Pxx_den = signal.welch(x, fs, nperseg=4096)
plt.plot(f, 10*np.log10(Pxx_den)); plt.xlabel('Frequency [Hz]'); plt.ylabel('PSD [V**2/Hz]'); plt.xlim([0,900]);  plt.grid();
plt.show()
Audio(x,rate=fs,autoplay=False,normalize=False)

## Filtr (trakt głosowy)

Trakt głosowy złożony z gardła, jamy ustnej i nosowej tworzą zespół filtrów pasmowo-przepustowych o wyraźnych częstotliwościach środkowych, nazywanych częstotliwościami rezonatorów. Odpowiednie ustawienie rezonatorów powinno zatem umożliwlić usłyszenie poszczególnych samogłosek. Bardzo dobra prezentacja działania modelu źródło filtr  jest przedstawiona w materiałach nauczyciela śpiewu Kena Bozemana:
[http://www.kenbozeman.com/power-spectrum-a.php](http://www.kenbozeman.com/power-spectrum-a.php).

Klasa FormantFIlter w poniższym bloku kodu zawiera podstawową implementację uproszczonego modelu traktu głosowego opartego na 4 filtrach IIR typu peak. Poza samymi filtrami klasa udostępnia metody do:
* filtracji sygnału
* wyświetlenia charakterystyki wypadkowej filtru oraz
* projektowania na podstawie zadanych częstotliwości f1,f2,f3,f4


In [ ]:
class FormantFilter:

    fs = 16000
    iir_filters = [
        (1,1)       # lis of (b,a) filters corresponding to formants
    ]

    def __init__(self, iir_filters = None, fs = 16000):
        self.fs = fs
        if iir_filters is not None:
            self.iir_filters = iir_filters

    @staticmethod
    def hiir(f,Q,fs):
        b, a = signal.iirpeak(f, Q, fs)
        freq, H = signal.freqz(b, a, fs=fs)
        return b, a, H, freq

    def design(
        self,
        filter_peak_freqs = [200, 700, 1000, 2000],
        gains = [1, 0.8, 0.1, 0.3],
        plot = True,
        quality = 15.0
    ):
        N = 16    # IIR filter order

        self.iir_filters = list()
        H_prot = 0

        assert len(filter_peak_freqs) == len(gains), \
            "Frequencies and gains vectors have different lengths."
        for f, g in zip(filter_peak_freqs, gains):
            b, a, H_temp, freq = self.hiir(f, quality, self.fs)
            self.iir_filters.append((g*b, a))
            H_prot += g * H_temp


        if plot:
            self.plot_filter(H_prot, freq)

    def filter(self, x):
        y = np.zeros_like(x)
        for b,a in self.iir_filters:
            y += signal.filtfilt(b,a,x)
        return y

    @staticmethod
    def plot_filter(h, freq):   # Plot
        plt.figure()
        fig, ax = plt.subplots(2, 1, figsize=(10, 4))
        ax[0].plot(freq, 20*np.log10(np.maximum(abs(h), 1e-5)), color='yellow')
        ax[0].set_title("Frequency Response")
        ax[0].set_ylabel("Amplitude (dB)", color='yellow')
        ax[0].set_xlim([0, 6000])
        ax[0].set_ylim([-60, 10])
        ax[0].grid()
        ax[1].plot(freq, np.unwrap(np.angle(h))*180/np.pi, color='green')
        ax[1].set_ylabel("Angle (degrees)", color='green')
        ax[1].set_xlabel("Frequency (Hz)")
        ax[1].set_xlim([0, 6000])
        ax[1].set_yticks([-90, -60, -30, 0, 30, 60, 90])
        ax[1].set_ylim([-90, 90])
        ax[1].grid()
        plt.show()

#FF = FormantFilter()
#FF.design(filter_peak_freqs = [753, 1261, 2513,5422]) # Male
#FF.design(filter_peak_freqs = [999, 1545,2756, 3951]) # Female

## Dźwięk

In [ ]:
# Niezbędne zmienne
fs = 16000
t = np.arange(0,3-1/fs,1/fs)

# ==========  Typ źródła ==========
f0 = 200    # Ton podstawowy, F0
#x = signal.square(2 * np.pi * f0 * t)
#x = signal.sawtooth(2 * np.pi * f0 * t)
x = LF_glottal_pulse_diff_Vaseghi(t,f0,fs)

# ========== Projekt traktu glosowego ==========
FF = FormantFilter()
# FF.design([999,1545, 2756, 3951])  # Female
FF.design([753, 1261, 2513,5422]) # Male

# Filtracja
y = FF.filter(x)

# Wyniki
fvec, tvec, STFT = signal.stft(y, fs, nperseg=512, noverlap=160, nfft=512,)
plot_stft(tvec,fvec[0:192],STFT[0:192],"STFT")
Audio(y,rate=fs,autoplay=False)


## Zadanie 3

Korzystając z powyższej implementacji (możesz ją dowolnie modyfikować by uzyskać lepsze efekty audio):
1. (0.4p)stwórz sygnały poszczególnych głosek /u/,/o/,/a/,/e/,/y/ oraz /i/ przy założeniu, że osobą mówiącą jest kobieta oraz  mężczyzna (w sumie 12 sygnałów). Skorzystaj z tablic dostępnych w wykładzie/internecie w celu określenia częstotliwości środkowych rezonatorów. Zamieść widgety Audio umożliwiające odsłuchanie każdej z głosek osobno.
2. (0.4p) łącząc odpowiednie głoski (cross-fading na krawędziach nie jest obowiązkowy ale mile widzany) stwórz słowa 'ioio', 'iou', 'eiai'
3. (0.2-0.5p) opisz zmiany jakie wprowadziłeś w kodzie klasy FormantFilter poprawiające działanie modelu. Jeśli uznasz to za konieczne możesz zmodyfikować sygnał źródła.

## 3.1 Sygnały samogłosek

In [ ]:
# Miejsce na Twój kod do Zadania 3
from IPython.display import Markdown, display

vowels = ['u','o','a','e','y','i']

#częstotliwości środkowe rezonatorów z wykładu 1 - wstęp
female_vowel_formants = {
    "u":[324,731,3140,4000],
    "o":[641,1080,2793,3885],
    "a":[999,1545,2756,3951],
    "e":[625,2210,2979,3964],
    "y":[329,2335,3041,4214],
    "i":[247,2775,3510,4155]
}

male_vowel_formants = {
    "u":[280,652,2459,3232],
    "o":[538,883,2571,3326],
    "a":[738,883,2571,3326],
    "e":[553,1794,2561,3408],
    "y":[323,1996,2821,3422],
    "i":[213,2316,2971,3542]
}

## zapisane sygnały użyte w zadaniu 3.2
female_vowel_signal ={}
male_vowel_signal={}

fs = 16000
t = np.arange(0,3-1/fs,1/fs)

f0_female = 220
f0_male = 200

x_male = LF_glottal_pulse_diff_Vaseghi(t,f0_male,fs)
x_female = LF_glottal_pulse_diff_Vaseghi(t,f0_female,fs)

FF = FormantFilter()
show_plots=False


for vowel in vowels:
    FF.design(female_vowel_formants[vowel],plot=show_plots,quality=20)
    yf = FF.filter(x_female)
    female_vowel_signal[vowel] = yf

    display(Markdown(f"### Female vowel '{vowel}'"))
    display(Audio(yf, rate=fs, autoplay=False))
    
    FF.design(male_vowel_formants[vowel],plot=show_plots,quality=20)
    ym = FF.filter(x_male)
    male_vowel_signal[vowel] = ym

    display(Markdown(f"### Male vowel '{vowel}'"))
    display(Audio(ym, rate=fs, autoplay=False))    

## 3.2 Łączenie głosek

### Funkcje

In [ ]:
def cross_fading(s1, s2, fs=16000, dur=0.04):
    dur_samples = int(fs * dur)
    dur_samples = min(dur_samples, len(s1), len(s2))
    
    left = s1[:-dur_samples]
    right = s2[dur_samples:]
    
    fade_out = np.linspace(1.0, 0.0, dur_samples)
    fade_in = np.linspace(0.0, 1.0, dur_samples)
    
    fade_left = s1[-dur_samples:] * fade_out
    fade_right = s2[:dur_samples] * fade_in
    
    return np.concatenate((left, fade_left + fade_right, right))


def speak_vowels(text, fs = 16000, vowel_dur = 0.2, display_audio = True, crossfading = True, crossfade_dur = 0.05, voice = 'female' ):
    vowels_text = "".join(char for char in text if char in vowels)
    
    vowel_dur = vowel_dur + crossfade_dur
    vowel_samples = int(vowel_dur*fs)
    
    
    if voice == 'male':
        voice_signal = male_vowel_signal
    else:
        voice_signal = female_vowel_signal
    
    output = voice_signal[vowels_text[0]][:vowel_samples]
    
    for char in vowels_text[1:]:
        if crossfading:
            output = cross_fading(output,voice_signal[char][:vowel_samples],dur = crossfade_dur)
        else:
            output = np.concatenate((output, voice_signal[char][:vowel_samples]))
        
    
    if display_audio:
        display(Markdown(f"### '{vowels_text}'"))
        display(Audio(output, rate=fs, autoplay=False))
    
    return output
    

### Wywołania

In [ ]:
#test = speak_vowels("ateiou")

output1 = speak_vowels("ioio")
output2 = speak_vowels("iou", crossfading = False)
output3 = speak_vowels("eiai")


## 3.3 Zmiany

<font color=red>Jedyna zmiana w klasie FormantFilter to dodanie parametru 'quality' do funkcji 'Design()' który poprzednio był stałą.</color>

#### <font color=red> W zadaniu 3.2 dodany crossfading usuwa słyszalną zmianę głosek i pozwala na płynne przejście między samogłoskami.</color>

## ***Interaktywne widgety

Notebook umożliwia stosowanie interaktywnych widgetów do sterowania parametrami funkcji. Poniższy przykłąd demonstruje wywołanie takiego widgetu dla funkcji design (UWAGA: funkcja w tej wersji, mimo że projektuje filtry, nie modyfikuje obiektu klasy FormantFilter). Jeśli uważasz to za interesujące dodaj widgety do Zadania 3.1 i opisz ich użycie w punkcie 3.3

In [ ]:
# Colab Widget Test
from ipywidgets import interactive
import matplotlib.pyplot as plt
import numpy as np

FF = FormantFilter()

def design(f1=300,f2=500,f3=1500,f4=2000):
        global FF
        FF.design(filter_peak_freqs = [f1,f2,f3,f4])
        return FF


interactive_plot = interactive(design,
                               f1=(200,400),
                               f2=(400, 1000),
                               f3=(1200, 2000),
                               f4=(1500, 2500))
output = interactive_plot.children[-1]
output.layout.height = '350px'
interactive_plot


Poniższy fragment kodu ściąga i zapisuje plik audio w katalogu roboczym, a następnie tworzy widget do odsłuchu pliku oraz wyświetla jego przebieg czasowy.



In [ ]:
from pathlib import Path
from urllib.request import urlopen

file = Path("data/PW1M1A__.wav")
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://home.agh.edu.pl/~witkow/audio/PW1M1A__.wav"
    with urlopen(url, timeout=30) as response:
        audio_data = response.read()
    file.write_bytes(audio_data)

import scipy.io.wavfile as wf
from IPython.display import Audio
fs,x = wf.read(file)
x = x/(2**15)
t = np.linspace(0,len(x)/fs,len(x))

import matplotlib.pyplot as plt
plt.plot(t,x)
plt.xlabel('Time [s]')
plt.ylabel('Magnitude')

Audio(x,rate=fs,autoplay=False)

# Ocena
### * Każde zadanie oceniane jest w skali od 0 do 1 z gradacją 0.1, co pozwala zdobyć do 3 punktów w całym notatniku.
### * 1 termin tego zadania upływa **28.10.2026 o godz. 13:00**. Po skończonej pracy należy zapisać plik opcją `File > Save and pin revision`. Prowadzący oceniając odtworzy ostanią wersję w repozytorium notatnika sprzed daty oddania i ta wersja będzie podlegać ocenie.